# Project Assignment: Transfer Learning on Oxford Flowers 102 Dataset

**Objective:** Apply transfer learning using pre-trained CNNs (ResNet50, VGG16, MobileNetV2) to classify images from the Oxford Flowers 102 dataset and compare model performance.

**Dataset:** Oxford Flowers 102 (102 flower categories) from TensorFlow Datasets.

You will:
- Load and explore the dataset
- Preprocess images for each model
- Adapt and train ResNet50, VGG16, and MobileNetV2
- Evaluate on the test split
- Answer reflection questions and attempt optional tasks


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

# Load Oxford Flowers 102 dataset (train/validation/test provided)
dataset, info = tfds.load('oxford_flowers102:2.1.1', with_info=True, as_supervised=True)
train_ds = dataset['train']
val_ds = dataset['validation']
test_ds = dataset['test']

print(info)
print('Train samples:', info.splits['train'].num_examples)
print('Validation samples:', info.splits['validation'].num_examples)
print('Test samples:', info.splits['test'].num_examples)
print('Number of classes:', info.features['label'].num_classes)

# Visualize a few training examples
for image, label in train_ds.take(3):
    plt.imshow(image)
    plt.title(f'Label: {label.numpy()}')
    plt.axis('off')
    plt.show()


In [ ]:
# Data Preprocessing
IMG_SIZE = 224  # 224x224 for ResNet50/VGG16/MobileNetV2
BATCH_SIZE = 32
NUM_CLASSES = 102

# Preprocessing functions per model
from tensorflow.keras.applications import resnet50, vgg16, mobilenet_v2

@tf.function
def preprocess_common(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

train_ds_base = train_ds.map(preprocess_common, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_base = val_ds.map(preprocess_common, num_parallel_calls=tf.data.AUTOTUNE)
test_ds_base = test_ds.map(preprocess_common, num_parallel_calls=tf.data.AUTOTUNE)

train_ds_prep = train_ds_base.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds_prep = val_ds_base.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds_prep = test_ds_base.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Define callbacks used across trainings
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint_resnet = ModelCheckpoint('resnet50_best.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
checkpoint_vgg = ModelCheckpoint('vgg16_best.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
checkpoint_mnv2 = ModelCheckpoint('mobilenetv2_best.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

earlystop = EarlyStopping(monitor='val_accuracy', patience=3, mode='max', restore_best_weights=True, verbose=1)


In [ ]:
# ResNet50: build, compile, and train
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input
from tensorflow.keras.models import Model

base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_resnet.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = resnet50.preprocess_input(inputs)
x = base_resnet(x)
x = GlobalAveragePooling2D()(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model_resnet = Model(inputs, outputs)

model_resnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_resnet = model_resnet.fit(
    train_ds_prep,
    validation_data=val_ds_prep,
    epochs=15,
    callbacks=[checkpoint_resnet, earlystop]
)


In [ ]:
# VGG16: build, compile, and train
from tensorflow.keras.applications import VGG16

base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_vgg.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = vgg16.preprocess_input(inputs)
x = base_vgg(x)
x = GlobalAveragePooling2D()(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model_vgg = Model(inputs, outputs)

model_vgg.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_vgg = model_vgg.fit(
    train_ds_prep,
    validation_data=val_ds_prep,
    epochs=15,
    callbacks=[checkpoint_vgg, earlystop]
)


In [ ]:
# MobileNetV2: build, compile, and train
from tensorflow.keras.applications import MobileNetV2

base_mnv2 = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_mnv2.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = mobilenet_v2.preprocess_input(inputs)
x = base_mnv2(x)
x = GlobalAveragePooling2D()(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model_mnv2 = Model(inputs, outputs)

model_mnv2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_mnv2 = model_mnv2.fit(
    train_ds_prep,
    validation_data=val_ds_prep,
    epochs=15,
    callbacks=[checkpoint_mnv2, earlystop]
)


In [ ]:
# Evaluate models on the test set
loss_resnet, acc_resnet = model_resnet.evaluate(test_ds_prep)
loss_vgg, acc_vgg = model_vgg.evaluate(test_ds_prep)
loss_mnv2, acc_mnv2 = model_mnv2.evaluate(test_ds_prep)

print(f'ResNet50   - Loss: {loss_resnet:.4f}, Accuracy: {acc_resnet:.4f}')
print(f'VGG16      - Loss: {loss_vgg:.4f}, Accuracy: {acc_vgg:.4f}')
print(f'MobileNetV2- Loss: {loss_mnv2:.4f}, Accuracy: {acc_mnv2:.4f}')


In [ ]:
# Optional: Train on validation split as well (for demonstration)
# Note: Typically we only validate on validation split; training on it is optional/for experiments.
history_resnet_val = model_resnet.fit(val_ds_prep, epochs=3)
history_vgg_val = model_vgg.fit(val_ds_prep, epochs=3)
history_mnv2_val = model_mnv2.fit(val_ds_prep, epochs=3)


In [ ]:
# Optional: Fine-tuning top layers
# Unfreeze top layers and fine-tune with a lower learning rate

def unfreeze_and_finetune(model, base_model, layers_to_unfreeze=50, lr=1e-5):
    for layer in base_model.layers[-layers_to_unfreeze:]:
        layer.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Fine-tune ResNet50 (e.g., last 30 layers)
unfreeze_and_finetune(model_resnet, base_resnet, layers_to_unfreeze=30, lr=1e-5)
history_resnet_ft = model_resnet.fit(
    train_ds_prep, validation_data=val_ds_prep, epochs=5, callbacks=[earlystop]
)

# Fine-tune VGG16
unfreeze_and_finetune(model_vgg, base_vgg, layers_to_unfreeze=30, lr=1e-5)
history_vgg_ft = model_vgg.fit(
    train_ds_prep, validation_data=val_ds_prep, epochs=5, callbacks=[earlystop]
)

# Fine-tune MobileNetV2
unfreeze_and_finetune(model_mnv2, base_mnv2, layers_to_unfreeze=30, lr=1e-5)
history_mnv2_ft = model_mnv2.fit(
    train_ds_prep, validation_data=val_ds_prep, epochs=5, callbacks=[earlystop]
)
